# Distance to k-Nearest Pharmacies from SAL Centroids
## Euclidean and OSMnx Network Distance (Pedestrian + Drive), k = 3

**Tess Vu**

This notebook computes the distance from each Small Area Layer (SAL)
centroid to the **three nearest** geocoded pharmacies using three methods:

1. **Euclidean** (straight-line) distance via scipy KDTree.
2. **Pedestrian network** distance via OSMnx walk graph.
3. **Drive network** distance via OSMnx drive graph.

k=3 captures access redundancy and choice beyond simple reachability.
A SAL with one pharmacy at 2.9 km passes a 3 km threshold, but if that
pharmacy closes or stocks out, the community is isolated. Three nearest
pharmacies reveal whether fallback options exist.

**Compute note:** k=1 used multi-source Dijkstra (one pass from all
pharmacy nodes). k=3 requires single-source Dijkstra from each pharmacy
node individually, which is substantially slower. Checkpointing is built
in so interrupted runs can resume without losing progress.

## SETUP AND IMPORTS

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())))

import heapq
import pickle
import time

import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import osmnx as ox
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from matplotlib_scalebar.scalebar import ScaleBar
from scipy.spatial import cKDTree

from src.config import load_config
from src.crs import CRS_UTM35S, CRS_WGS84
from src.io import ensure_dir
from src.paths import (FIGURES, NETWORKS, PHARMACIES_MASTER, POP_PRED_FINAL,
                       SAL_PHARM_DIST_K1, SAL_PHARM_DIST_K3, SAL_W_WARD_DEDUP)
from src.plotting import FONT_BODY, FONT_TITLE, setup_fonts

print(f"OSMnx version: {ox.__version__}")
print(f"NetworkX version: {nx.__version__}")

OSMnx version: 2.1.0
NetworkX version: 3.6.1


In [ ]:
CFG = load_config()
K_NEAREST = CFG["k_nearest"]
CUTOFF_M = CFG["dijkstra_cutoff_m"]

IMAGE_DIR = ensure_dir(FIGURES / "sal_pharmacy_distance")
CHECKPOINT_DIR = ensure_dir(NETWORKS / "checkpoints")
CHECKPOINT_INTERVAL = 50

PROVINCES = ["Gauteng", "KwaZulu-Natal"]

GRAPHML_PATHS = {
    "Gauteng_walk": NETWORKS / "network_gauteng_walk.graphml",
    "Gauteng_drive": NETWORKS / "network_gauteng_drive.graphml",
    "KwaZulu-Natal_walk": NETWORKS / "network_kwazulu_natal_walk.graphml",
    "KwaZulu-Natal_drive": NETWORKS / "network_kwazulu_natal_drive.graphml",
}

print(f"K_NEAREST: {K_NEAREST}")
print(f"CUTOFF: {CUTOFF_M / 1000:.0f} km")
print(f"CHECKPOINT INTERVAL: every {CHECKPOINT_INTERVAL} pharmacies")

In [ ]:
setup_fonts()

# Low (navy) -> mid (teal) -> high (gold).
custom_cmap = LinearSegmentedColormap.from_list(
    "pharmacy_distance",
    ["#2e3e6c", "#007264", "#ffcb05"],
    N=256,
)

def reformat_legend_labels(ax):
    """'a, b' -> 'a - b'; '-inf, x' or 'x, x' -> 'x'."""
    legend = ax.get_legend()
    if legend is None:
        return
    for text in legend.get_texts():
        label = text.get_text()
        if ", " in label:
            lo, hi = (s.strip() for s in label.split(", ", 1))
            text.set_text(hi if lo == hi or "inf" in lo.lower() else f"{lo} - {hi}")

In [3]:
# Define custom color ramp for heatmaps.
# Low (teal) -> Mid (gold) -> High (rust).
custom_cmap = LinearSegmentedColormap.from_list(
    "pharmacy_distance",
    ["#2e3e6c", "#007264", "#ffcb05"],
    N = 256
)

def reformat_legend_labels(ax):
    """Reformat geopandas interval legend labels:
    'a, b' -> 'a - b', and collapse '-inf, x' or 'x, x' -> just 'x'.
    """
    legend = ax.get_legend()
    if legend is None:
        return
    for text in legend.get_texts():
        label = text.get_text()
        if ", " in label:
            parts = label.split(", ", 1)
            lo, hi = parts[0].strip(), parts[1].strip()
            if lo == hi or "inf" in lo.lower():
                text.set_text(hi)
            else:
                text.set_text(f"{lo} - {hi}")


## CONFIGURATION

All file paths, distance parameters, and checkpoint settings
in a single cell. Update paths to match local directory structure.

In [ ]:
# FILE PATHS
# SAL shapefile with ward assignments and geometry.
SAL_W_WARD_DEDUP = "data/sal_w_ward_dedup/sal_w_ward_dedup.shp"

# Population estimates from the step-down and dasymetric notebook.
POP_ESTIMATES_PATH = "data/pop_pred_final.csv"

# Fully geocoded pharmacy file with lat and lng columns.
PHARMACIES_PATH = "data/PHARMACIES_MASTER_FINAL.csv"

# Pre-saved network graph directory.
NETWORK_DIR = "data/networks"

# Output directory for results and checkpoints.
OUTPUT_DIR = "data/networks"
os.makedirs(OUTPUT_DIR, exist_ok = True)

# Image output directory.
IMAGE_DIR = "images/sal_pharmacy_distance"
os.makedirs(IMAGE_DIR, exist_ok = True)

# Projected CRS for distance calculations in meters.
PROJECTED_CRS = "EPSG:32735"

# Province names for filtering and labeling.
PROVINCES = ["Gauteng", "KwaZulu-Natal"]

# K-NEAREST PARAMETER
# Number of nearest pharmacies to compute for each SAL centroid.
K_NEAREST = 3

# NETWORK DISTANCE CUTOFF (meters)
# Dijkstra stops exploring beyond this distance from each pharmacy node.
# 50 km is generous; keeps compute bounded without missing realistic access.
CUTOFF_M = 50000

# CHECKPOINT SETTINGS
# Directory for intermediate checkpoint files.
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok = True)

# Save checkpoint after every N pharmacy nodes processed.
CHECKPOINT_INTERVAL = 50

# Expected graphml file paths (must already exist on disk).
GRAPHML_PATHS = {
    "Gauteng_walk": os.path.join(NETWORK_DIR, "network_gauteng_walk.graphml"),
    "Gauteng_drive": os.path.join(NETWORK_DIR, "network_gauteng_drive.graphml"),
    "KwaZulu-Natal_walk": os.path.join(NETWORK_DIR, "network_kwazulu_natal_walk.graphml"),
    "KwaZulu-Natal_drive": os.path.join(NETWORK_DIR, "network_kwazulu_natal_drive.graphml"),
}

print(f"K_NEAREST: {K_NEAREST}")
print(f"CUTOFF: {CUTOFF_M / 1000:.0f} km")
print(f"CHECKPOINT INTERVAL: every {CHECKPOINT_INTERVAL} pharmacies")
print(f"OUTPUT DIR: {os.path.abspath(OUTPUT_DIR)}")

K_NEAREST: 3
CUTOFF: 50 km
CHECKPOINT INTERVAL: every 50 pharmacies
OUTPUT DIR: c:\Users\Tess\Desktop\UPenn\UPenn_SS26\MUSA_8010-001_Practicum\south-africa-healthcare\notebooks\data\networks


## LOAD SAL GEOMETRIES AND POPULATION ESTIMATES

Load the SAL shapefile that has ward assignments, then join
with the population estimates produced by the dasymetric notebook.

In [ ]:
# Load SAL shapefile with geometry and ward assignments.
sal_geo = gpd.read_file(SAL_W_WARD_DEDUP)
print(f"SAL SHAPEFILE LOADED: {sal_geo.shape[0]} features")
print(f"CRS: {sal_geo.crs}")
print(f"Columns: {list(sal_geo.columns)}")

SAL SHAPEFILE LOADED: 38380 features
CRS: EPSG:32735
Columns: ['SP_CODE', 'SP_NAME', 'MP_CODE', 'MP_NAME', 'MN_MDB_C', 'MN_CODE', 'MN_NAME', 'MN_TYPE', 'DC_MDB_C', 'DC_MN_C', 'DC_NAME', 'PR_MDB_C', 'PR_CODE', 'PR_NAME', 'EA_GTYPE', 'ALBERS_ARE', 'MD_CODE', 'MD_NAME', 'Shape_Leng', 'SAL_CODE', 'EA_TYPE', 'F4_class', 'EA_area_km', 'num_houses', 'F4_class_2', 'num_build', 'EA_CODE_1', 'old_EA_TYP', 'smallplace', 'url', 'Black_Afri', 'White', 'Coloured', 'Indian_or', 'Other', 'population', 'F0_4', 'F5_9', 'F10_14', 'F15_19', 'F20_24', 'F25_29', 'F30_34', 'F35_39', 'F40_44', 'F45_49', 'F50_54', 'F55_59', 'F60_64', 'F65_69', 'F70_74', 'F75_79', 'F80_84', 'F85_', 'Shape_Le_1', 'Shape_Area', 'sal_pop_de', 'sal2011_po', 'OBJECTID_1', 'EA_CODE', 'census_war', 'AREA', 'PERCENTAGE', 'OBJECTID_2', 'EA_CODE_12', 'OBJECTID_3', 'FREQUENCY', 'EA_CODE_13', 'MAX_AREA', 'geometry']


In [6]:
# Load population estimates from the dasymetric step-down.
pop_est = pd.read_csv(POP_ESTIMATES_PATH)
print(f"POPULATION ESTIMATES LOADED: {pop_est.shape[0]} rows")
print(f"Columns: {list(pop_est.columns)}")

POPULATION ESTIMATES LOADED: 38380 rows
Columns: ['WardID', 'EA_CODE', 'sal2011_pop', 'ward2023_pop', 'EA_GTYPE', 'EA_TYPE', 'econ_status', 'houses2011', 'Black_Afri', 'White', 'Coloured', 'Indian_or', 'Other', 'area_km2', 'sal_dense', 'log_density', 'ward2011_sum', 'share2011', 'dasym_weight', 'sal2023_est', 'growth_rate']


In [7]:
# Join population estimates onto SAL geometries.
# The join key is EA_CODE (SAL identifier).
sal = sal_geo.merge(
    pop_est[["EA_CODE", "sal2023_est", "WardID"]],
    on = "EA_CODE",
    how = "left"
)

# Report join quality.
matched = sal["sal2023_est"].notna().sum()
total = sal.shape[0]
print(f"JOIN RESULTS: {matched}/{total} SALs matched ({matched / total * 100:.1f}%)")

# Drop SALs with no population estimate.
sal = sal[sal["sal2023_est"].notna()].copy()
print(f"SALs RETAINED AFTER DROPPING UNMATCHED: {sal.shape[0]}")

JOIN RESULTS: 38380/38380 SALs matched (100.0%)
SALs RETAINED AFTER DROPPING UNMATCHED: 38380


## COMPUTE SAL CENTROIDS

Project to a metric CRS, compute geometric centroids, then extract
coordinates for distance calculations.

**Limitation:** Geometric centroids may fall in uninhabited areas
(rivers, parks, industrial zones) for irregularly shaped SALs.
Population-weighted centroids using building footprints would be
more defensible but require additional data processing.

In [8]:
# Project SAL geometries to metric CRS for distance calculations.
sal_proj = sal.to_crs(PROJECTED_CRS)

# Compute geometric centroids in projected space.
sal_proj["centroid_geom"] = sal_proj.geometry.centroid

# Extract centroid coordinates in projected CRS (meters).
sal_proj["centroid_x"] = sal_proj["centroid_geom"].x
sal_proj["centroid_y"] = sal_proj["centroid_geom"].y

# Also compute centroids in WGS84 for OSMnx (which expects lat/lng).
sal_wgs84 = sal.to_crs("EPSG:4326")
sal_wgs84["centroid_wgs84"] = sal_wgs84.geometry.centroid
sal_proj["centroid_lat"] = sal_wgs84["centroid_wgs84"].y.values
sal_proj["centroid_lng"] = sal_wgs84["centroid_wgs84"].x.values

print(f"SAL CENTROIDS COMPUTED: {sal_proj.shape[0]} centroids")
print(f"Projected CRS: {PROJECTED_CRS}")
print(f"Centroid X range: {sal_proj['centroid_x'].min():.0f} to {sal_proj['centroid_x'].max():.0f} m")
print(f"Centroid Y range: {sal_proj['centroid_y'].min():.0f} to {sal_proj['centroid_y'].max():.0f} m")

SAL CENTROIDS COMPUTED: 38380 centroids
Projected CRS: EPSG:32735
Centroid X range: 525554 to 1084240 m
Centroid Y range: 6557451 to 7210624 m


## LOAD GEOCODED PHARMACIES

Load the fully geocoded pharmacy file and convert to a GeoDataFrame.

In [9]:
# Load geocoded pharmacies.
pharm_df = pd.read_csv(PHARMACIES_PATH)
print(f"PHARMACIES LOADED: {pharm_df.shape[0]} rows")
print(f"Columns: {list(pharm_df.columns)}")

# Check for lat/lng columns and filter valid coordinates.
assert "LAT" in pharm_df.columns, "Missing 'LAT' column in pharmacy file."
assert "LNG" in pharm_df.columns, "Missing 'LNG' column in pharmacy file."

# Drop rows with missing coordinates.
valid_coords = pharm_df["LAT"].notna() & pharm_df["LNG"].notna()
print(f"Pharmacies with valid coordinates: {valid_coords.sum()}/{pharm_df.shape[0]}")
pharm_df = pharm_df[valid_coords].copy()

# Convert to GeoDataFrame.
pharm_gdf = gpd.GeoDataFrame(
    pharm_df,
    geometry = gpd.points_from_xy(pharm_df["LNG"], pharm_df["LAT"]),
    crs = "EPSG:4326"
)

# Project to metric CRS.
pharm_proj = pharm_gdf.to_crs(PROJECTED_CRS)
pharm_proj["pharm_x"] = pharm_proj.geometry.x
pharm_proj["pharm_y"] = pharm_proj.geometry.y

print(f"PHARMACIES READY: {pharm_proj.shape[0]} geocoded locations")

PHARMACIES LOADED: 2241 rows
Columns: ['RECORD_ID', 'Y_NUMBER', 'PHARMACY_ID', 'PLACE_ID', 'NAME', 'STATUS', 'LICENCE_NUMBER', 'REGISTRATION_DATE', 'OWNER', 'INSPECTION', 'ADDRESS', 'CITY', 'PROVINCE', 'TELEPHONE', 'MATCHED_NAME', 'MATCHED_ADDRESS', 'LAT', 'LNG', 'TYPES', 'OPENING_HOURS', 'SOURCE', 'SPATIAL_CHECK']
Pharmacies with valid coordinates: 2241/2241
PHARMACIES READY: 2241 geocoded locations


In [10]:
# Quick sanity check: bounding box of pharmacies vs SALs.
print("BOUNDING BOX COMPARISON (WGS84)")
sal_bounds = sal.to_crs("EPSG:4326").total_bounds
pharm_bounds = pharm_gdf.total_bounds
print(f"SALs: W={sal_bounds[0]:.3f}, S={sal_bounds[1]:.3f}, E={sal_bounds[2]:.3f}, N={sal_bounds[3]:.3f}")
print(f"Pharmacies: W={pharm_bounds[0]:.3f}, S={pharm_bounds[1]:.3f}, E={pharm_bounds[2]:.3f}, N={pharm_bounds[3]:.3f}")

BOUNDING BOX COMPARISON (WGS84)
SALs: W=27.156, S=-31.083, E=32.891, N=-25.110
Pharmacies: W=17.989, S=-34.091, E=32.757, N=-23.697


## EUCLIDEAN DISTANCE TO k-NEAREST PHARMACIES

Use a KDTree for efficient k-nearest-neighbor lookup in projected
coordinates (meters). This gives straight-line distances that
underestimate true travel distances but serve as a useful lower bound.
KDTree handles k > 1 natively with negligible additional cost.

In [11]:
# Build KDTree from pharmacy locations in projected CRS.
pharm_coords = np.column_stack([pharm_proj["pharm_x"].values, pharm_proj["pharm_y"].values])
tree = cKDTree(pharm_coords)

# Query k nearest pharmacies for each SAL centroid.
sal_coords = np.column_stack([sal_proj["centroid_x"].values, sal_proj["centroid_y"].values])
distances_m, indices = tree.query(sal_coords, k = K_NEAREST)

# Store results for each k level.
for ki in range(K_NEAREST):
    col_m = f"euclidean_dist_k{ki + 1}_m"
    col_km = f"euclidean_dist_k{ki + 1}_km"
    sal_proj[col_m] = distances_m[:, ki]
    sal_proj[col_km] = distances_m[:, ki] / 1000.0

print(f"EUCLIDEAN DISTANCES COMPUTED (k = {K_NEAREST})")
print()
for ki in range(K_NEAREST):
    col_km = f"euclidean_dist_k{ki + 1}_km"
    print(f"k={ki + 1}:")
    print(sal_proj[col_km].describe().to_string())
    print()

EUCLIDEAN DISTANCES COMPUTED (k = 3)

k=1:
count    38380.000000
mean         4.427563
std          6.919707
min          0.005177
25%          0.655182
50%          1.341396
75%          4.197295
max         44.852519

k=2:
count    38380.000000
mean         5.895443
std          8.420185
min          0.036628
25%          1.071998
50%          2.011297
75%          6.161883
max         60.477002

k=3:
count    38380.000000
mean         7.200930
std         10.232959
min          0.061974
25%          1.378255
50%          2.477753
75%          7.559585
max         79.956276



## EUCLIDEAN DISTANCE HEATMAP

Choropleth map of SAL polygons colored by Euclidean distance
to the nearest pharmacy (k=1), shown per province.

In [12]:
# Prepare data for mapping (keep original polygon geometry, not centroids).
sal_map_data = sal_proj.copy()
sal_map_data = sal_map_data.set_geometry("geometry")

# Determine province column name from shapefile.
prov_col = None
for candidate in ["PR_NAME", "PROVINCE", "Province", "province"]:
    if candidate in sal_map_data.columns:
        prov_col = candidate
        break

if prov_col is None:
    print("WARNING: No province column found. Plotting all SALs together.")
    prov_col = "ALL"
    sal_map_data["ALL"] = "All Provinces"

print(f"Province column: {prov_col}")
print(f"Unique values: {sal_map_data[prov_col].unique()}")

Province column: PR_NAME
Unique values: <ArrowStringArray>
['KwaZulu-Natal', 'Gauteng']
Length: 2, dtype: str


In [ ]:
# Define distance bins in kilometers.
dist_bins_km = [0, 1, 2, 3, 5, 10, 15, 25, 50]

fig, axes = plt.subplots(1, 2, figsize = (18, 10))
cmap = custom_cmap

provinces_in_data = sal_map_data[prov_col].unique()

for i, prov in enumerate(provinces_in_data[:2]):
    subset = sal_map_data[sal_map_data[prov_col] == prov].copy()
    subset = subset[subset["euclidean_dist_k1_km"].notna()]

    subset.plot(
        column = "euclidean_dist_k1_km",
        cmap = cmap,
        scheme = "UserDefined",
        classification_kwds = {"bins": dist_bins_km},
        legend = True,
        legend_kwds = {"title": "Distance (km)", "loc": "lower right", "fmt": "{:.0f}"},
        ax = axes[i],
        edgecolor = "none",
        linewidth = 0
    )

    axes[i].get_legend().get_title().set_fontweight("bold")
    reformat_legend_labels(axes[i])
    axes[i].set_title(f"{prov}: Euclidean Distance to Nearest Pharmacy (k = 1)", fontweight = "bold")
    
    # Add scalebar.
    scalebar = ScaleBar(1, units = "m", location = "lower left", box_alpha = 0.8, font_properties = {"size": 10})
    axes[i].add_artist(scalebar)
    axes[i].axis("off")

plt.suptitle("SAL Centroid Euclidean Distance to Nearest Pharmacy", y = 1.02, size = 14, fontweight = "bold")
plt.tight_layout()

output_path = os.path.join(IMAGE_DIR, "heatmap_euclidean_k1.png")

plt.savefig(output_path, dpi = 300, bbox_inches = "tight")
plt.savefig(f"{output_path}_transparent.png", dpi = 300, bbox_inches = "tight", transparent = True)
print(f"SAVED: {output_path}")
plt.show()

## LOAD SAVED NETWORK GRAPHS

Load pre-saved `.graphml` files from disk. These were downloaded
in a previous session and should not need re-downloading.

**Expected files:**
- `network_gauteng_walk.graphml`
- `network_gauteng_drive.graphml`
- `network_kwazulu_natal_walk.graphml`
- `network_kwazulu_natal_drive.graphml`

In [14]:
# Verify all expected graphml files exist before loading.
missing = []
for key, path in GRAPHML_PATHS.items():
    exists = os.path.exists(path)
    status = "FOUND" if exists else "MISSING"
    print(f"  {status}: {path}")
    if not exists:
        missing.append(key)

if missing:
    print(f"MISSING GRAPHS: {missing}")
    print("Cannot proceed without all network graphs.")
    print("Re-run the download step or check file paths.")
else:
    print("ALL NETWORK GRAPHS FOUND")

  FOUND: data/networks\network_gauteng_walk.graphml
  FOUND: data/networks\network_gauteng_drive.graphml
  FOUND: data/networks\network_kwazulu_natal_walk.graphml
  FOUND: data/networks\network_kwazulu_natal_drive.graphml
ALL NETWORK GRAPHS FOUND


In [15]:
# Load all network graphs from disk.
# This is slow (large files) but much faster than re-downloading.
graphs = {}

for key, path in GRAPHML_PATHS.items():
    if not os.path.exists(path):
        print(f"SKIPPING {key}: file not found.")
        continue

    print(f"LOADING: {key}")
    start_time = time.time()
    G = ox.load_graphml(path)
    elapsed = time.time() - start_time
    print(f"  Loaded in {elapsed:.1f}s: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")
    graphs[key] = G

print(f"GRAPHS LOADED: {list(graphs.keys())}")

LOADING: Gauteng_walk
  Loaded in 86.2s: 443,832 nodes, 1,219,312 edges
LOADING: Gauteng_drive
  Loaded in 61.1s: 279,004 nodes, 730,792 edges
LOADING: KwaZulu-Natal_walk
  Loaded in 114.7s: 541,204 nodes, 1,356,318 edges
LOADING: KwaZulu-Natal_drive
  Loaded in 74.1s: 311,426 nodes, 749,194 edges
GRAPHS LOADED: ['Gauteng_walk', 'Gauteng_drive', 'KwaZulu-Natal_walk', 'KwaZulu-Natal_drive']


## NETWORK DISTANCE COMPUTATION (k = 3)

**Why k=3 changes the algorithm:**

k=1 used multi-source Dijkstra, seeding from all pharmacy nodes at
once. That gives the minimum distance from *any* pharmacy to each
graph node in a single pass, which is very fast, but it cannot
distinguish which pharmacy is nearest, second-nearest, or third.

k=3 requires single-source Dijkstra from **each pharmacy node
individually**, then accumulating the k smallest distances per SAL
centroid node. This is O(P x V log V) where P = pharmacy nodes
and V = reachable graph nodes within the cutoff.

**Checkpointing:** Progress is saved to disk every N pharmacies.
If the kernel crashes or is interrupted, re-running the cell will
resume from the last checkpoint rather than starting over.

**Memory management:** A max-heap of fixed size k is maintained per
SAL centroid node, so memory is bounded at O(SAL_nodes x k)
regardless of how many pharmacies are processed.

In [16]:
def compute_network_distances_kn(G, sal_df, pharm_df, k = 3,
                                 cutoff_m = 50000, label = "network",
                                 checkpoint_dir = "data/networks/checkpoints",
                                 checkpoint_interval = 50):
    """Compute shortest-path distances from each SAL centroid to
    the k nearest pharmacies using per-pharmacy Dijkstra with
    checkpointing.

    Parameters
    G : networkx.MultiDiGraph (unprojected, WGS84).
    sal_df : GeoDataFrame with centroid_lat and centroid_lng.
    pharm_df : GeoDataFrame with lat and lng.
    k : number of nearest pharmacies to track per SAL.
    cutoff_m : Dijkstra exploration cutoff in meters.
    label : string label for checkpoint filename.
    checkpoint_dir : directory for checkpoint pickle files.
    checkpoint_interval : save checkpoint every N pharmacies.

    Returns
    np.ndarray of shape (len(sal_df), k) with distances in meters.
    NaN where fewer than k pharmacies are reachable.
    """
    os.makedirs(checkpoint_dir, exist_ok = True)
    checkpoint_path = os.path.join(checkpoint_dir, f"checkpoint_{label}_k{k}.pkl")

    # Snap pharmacies to nearest graph nodes.
    pharm_nodes_all = ox.nearest_nodes(
        G, X = pharm_df["LNG"].values, Y = pharm_df["LAT"].values
    )
    unique_pharm_nodes = list(set(pharm_nodes_all))
    print(f"  Pharmacy nodes snapped: {len(pharm_nodes_all)} total -> {len(unique_pharm_nodes)} unique")

    # Snap SAL centroids to nearest graph nodes.
    sal_nodes = ox.nearest_nodes(
        G, X = sal_df["centroid_lng"].values, Y = sal_df["centroid_lat"].values
    )
    unique_sal_nodes = set(sal_nodes)
    print(f"  SAL centroid nodes snapped: {len(sal_nodes)} total, {len(unique_sal_nodes)} unique")

    # Convert to undirected for bidirectional traversal.
    G_undirected = G.to_undirected()

    # Load checkpoint if one exists (resume interrupted run).
    if os.path.exists(checkpoint_path):
        with open(checkpoint_path, "rb") as f:
            checkpoint = pickle.load(f)
        processed = checkpoint["processed"]
        sal_heaps = checkpoint["sal_heaps"]
        print(f"  CHECKPOINT LOADED: {len(processed)}/{len(unique_pharm_nodes)} pharmacies already processed")
    else:
        processed = set()
        # Max-heap of size k per unique SAL node.
        # Store negative distances so heapq (min-heap) acts as max-heap.
        sal_heaps = {node: [] for node in unique_sal_nodes}

    remaining = [n for n in unique_pharm_nodes if n not in processed]
    total_remaining = len(remaining)
    total_pharm = len(unique_pharm_nodes)

    if total_remaining == 0:
        print(f"  ALL {total_pharm} PHARMACIES ALREADY PROCESSED (checkpoint complete)")
    else:
        print(f"  PROCESSING {total_remaining} remaining pharmacies (of {total_pharm} total)")

    # Track timing for estimates.
    batch_start = time.time()
    cumulative_dijkstra_time = 0.0

    for idx, pharm_node in enumerate(remaining):
        # Single-source Dijkstra with cutoff from this pharmacy node.
        t0 = time.time()
        dist_dict = nx.single_source_dijkstra_path_length(
            G_undirected, pharm_node, cutoff = cutoff_m, weight = "length"
        )
        cumulative_dijkstra_time += time.time() - t0

        # Update heaps for SAL nodes reached by this pharmacy.
        for sal_node in unique_sal_nodes:
            if sal_node in dist_dict:
                d = dist_dict[sal_node]
                heap = sal_heaps[sal_node]
                if len(heap) < k:
                    # Heap not full yet, push directly.
                    heapq.heappush(heap, -d)
                elif d < -heap[0]:
                    # New distance is smaller than current max in heap, replace.
                    heapq.heapreplace(heap, -d)

        processed.add(pharm_node)

        # Progress reporting every checkpoint_interval pharmacies.
        done_count = idx + 1
        if done_count % checkpoint_interval == 0 or done_count == total_remaining:
            elapsed = time.time() - batch_start
            avg_per_pharm = elapsed / done_count
            remaining_count = total_remaining - done_count
            eta_seconds = avg_per_pharm * remaining_count
            eta_minutes = eta_seconds / 60

            print(f"  PROGRESS: {len(processed)}/{total_pharm} pharmacies "
                  f"({done_count}/{total_remaining} this session) | "
                  f"avg {avg_per_pharm:.1f}s/pharm | "
                  f"ETA: {eta_minutes:.0f} min")

            # Save checkpoint.
            with open(checkpoint_path, "wb") as f:
                pickle.dump({"processed": processed, "sal_heaps": sal_heaps}, f)

    total_elapsed = time.time() - batch_start
    print(f"  DIJKSTRA COMPLETE: {total_elapsed / 60:.1f} min total, "
          f"{cumulative_dijkstra_time / 60:.1f} min in Dijkstra")

    # Extract top-k distances for each SAL centroid (in original order).
    results = np.full((len(sal_nodes), k), np.nan)
    for i, sal_node in enumerate(sal_nodes):
        # Heap contains negative distances; negate and sort ascending.
        dists_neg = sal_heaps.get(sal_node, [])
        dists = sorted([-d for d in dists_neg])
        for j in range(min(k, len(dists))):
            results[i, j] = dists[j]

    # Report reachability per k level.
    for ki in range(k):
        reachable = np.isfinite(results[:, ki]).sum()
        print(f"  k={ki + 1}: {reachable}/{len(sal_nodes)} SALs reachable "
              f"({reachable / len(sal_nodes) * 100:.1f}%)")

    # Clean up checkpoint file after successful completion.
    if os.path.exists(checkpoint_path):
        os.remove(checkpoint_path)
        print(f"  CHECKPOINT CLEANED UP: {checkpoint_path}")

    return results

## COMPUTE TIME ESTIMATES

Estimate total run time before starting the computation.
Rough heuristic: ~2-10 seconds per pharmacy node per Dijkstra,
depending on graph density and cutoff distance.

In [17]:
# Estimate run time for each province x mode combination.
# Determine province column.
if prov_col is None:
    for candidate in ["PR_NAME", "PROVINCE", "Province", "province"]:
        if candidate in sal_proj.columns:
            prov_col = candidate
            break

# Rough seconds-per-pharmacy heuristic based on graph node count.
# Walk graphs are denser (more nodes) and slower than drive graphs.
def estimate_seconds_per_pharmacy(n_nodes):
    """Rough heuristic for Dijkstra time per pharmacy node."""
    if n_nodes > 1_000_000:
        return 8.0
    elif n_nodes > 500_000:
        return 5.0
    elif n_nodes > 100_000:
        return 3.0
    else:
        return 1.5

print("ESTIMATED RUN TIMES")
print(f"(Assumes {CUTOFF_M / 1000:.0f} km cutoff, k = {K_NEAREST})")
print()

total_estimate_min = 0

for prov_name in PROVINCES:
    for net_type in ["walk", "drive"]:
        graph_key = f"{prov_name}_{net_type}"
        if graph_key not in graphs:
            print(f"  {graph_key}: GRAPH NOT LOADED, skipping estimate")
            continue

        G = graphs[graph_key]
        n_nodes = G.number_of_nodes()

        # Count pharmacies for this province.
        pharm_prov_col = None
        for candidate in ["PROVINCE", "Province", "province"]:
            if candidate in pharm_gdf.columns:
                pharm_prov_col = candidate
                break

        if pharm_prov_col is not None:
            prov_mask = pharm_gdf[pharm_prov_col].str.contains(prov_name, case = False, na = False)
            n_pharm = prov_mask.sum()
        else:
            # Rough count from spatial bounds.
            prov_mask_sal = sal_proj[prov_col].str.contains(prov_name, case = False, na = False)
            prov_bounds = sal_proj[prov_mask_sal].to_crs("EPSG:4326").total_bounds
            n_pharm = len(pharm_gdf.cx[prov_bounds[0]:prov_bounds[2], prov_bounds[1]:prov_bounds[3]])

        sec_per = estimate_seconds_per_pharmacy(n_nodes)
        est_min = (n_pharm * sec_per) / 60

        # Check for existing checkpoint (already-processed pharmacies).
        cp_path = os.path.join(CHECKPOINT_DIR, f"checkpoint_{prov_name}_{net_type}_k{K_NEAREST}.pkl")
        already_done = 0
        if os.path.exists(cp_path):
            with open(cp_path, "rb") as f:
                cp = pickle.load(f)
            already_done = len(cp["processed"])

        remaining_pharm = max(0, n_pharm - already_done)
        adj_est_min = (remaining_pharm * sec_per) / 60

        total_estimate_min += adj_est_min

        checkpoint_note = f" ({already_done} already checkpointed)" if already_done > 0 else ""
        print(f"  {graph_key}:")
        print(f"    Graph nodes: {n_nodes:,}")
        print(f"    Pharmacies: ~{n_pharm}{checkpoint_note}")
        print(f"    Est. {sec_per:.0f}s/pharmacy -> ~{adj_est_min:.0f} min")
        print()

print(f"TOTAL ESTIMATED TIME: ~{total_estimate_min:.0f} min ({total_estimate_min / 60:.1f} hours)")
print()
print("These are rough estimates. Actual times depend on graph density,")
print("cutoff distance, and hardware. The checkpoint system ensures no")
print("progress is lost if the kernel is interrupted.")

ESTIMATED RUN TIMES
(Assumes 50 km cutoff, k = 3)

  Gauteng_walk:
    Graph nodes: 443,832
    Pharmacies: ~1511
    Est. 3s/pharmacy -> ~76 min

  Gauteng_drive:
    Graph nodes: 279,004
    Pharmacies: ~1511
    Est. 3s/pharmacy -> ~76 min

  KwaZulu-Natal_walk:
    Graph nodes: 541,204
    Pharmacies: ~70
    Est. 5s/pharmacy -> ~6 min

  KwaZulu-Natal_drive:
    Graph nodes: 311,426
    Pharmacies: ~70
    Est. 3s/pharmacy -> ~4 min

TOTAL ESTIMATED TIME: ~160 min (2.7 hours)

These are rough estimates. Actual times depend on graph density,
cutoff distance, and hardware. The checkpoint system ensures no
progress is lost if the kernel is interrupted.


## RUN k=3 NETWORK DISTANCE COMPUTATION

This is the long-running step. Each province x mode combination
runs independently with its own checkpoint file.

**If interrupted:** Re-run this cell. It will detect existing
checkpoints and resume from where it left off.

**After completion:** Checkpoint files are automatically deleted.

In [18]:
# Compute k-nearest network distances for each province and network type.
for prov_name in PROVINCES:
    # Filter SALs for this province.
    prov_mask_sal = sal_proj[prov_col].str.contains(prov_name, case = False, na = False)
    sal_prov = sal_proj[prov_mask_sal].copy()

    if sal_prov.shape[0] == 0:
        print(f"WARNING: No SALs for {prov_name}. Check province column values.")
        continue

    # Filter pharmacies for this province.
    pharm_prov_col = None
    for candidate in ["PROVINCE", "Province", "province"]:
        if candidate in pharm_gdf.columns:
            pharm_prov_col = candidate
            break

    if pharm_prov_col is not None:
        prov_mask_pharm = pharm_gdf[pharm_prov_col].str.contains(prov_name, case = False, na = False)
        pharm_prov = pharm_gdf[prov_mask_pharm].copy()
    else:
        # Fallback: spatial bounding box filter.
        prov_bounds = sal_prov.to_crs("EPSG:4326").total_bounds
        pharm_prov = pharm_gdf.cx[prov_bounds[0]:prov_bounds[2], prov_bounds[1]:prov_bounds[3]].copy()

    print(f"PROVINCE: {prov_name}")
    print(f"  SALs: {sal_prov.shape[0]}, Pharmacies: {pharm_prov.shape[0]}")

    for net_type in ["walk", "drive"]:
        graph_key = f"{prov_name}_{net_type}"
        if graph_key not in graphs:
            print(f"  SKIPPING {graph_key}: graph not loaded.")
            continue

        G = graphs[graph_key]
        label = f"{prov_name}_{net_type}"

        print(f"  COMPUTING {net_type.upper()} NETWORK DISTANCES (k = {K_NEAREST})")
        run_start = time.time()

        results = compute_network_distances_kn(
            G = G,
            sal_df = sal_prov,
            pharm_df = pharm_prov,
            k = K_NEAREST,
            cutoff_m = CUTOFF_M,
            label = label,
            checkpoint_dir = CHECKPOINT_DIR,
            checkpoint_interval = CHECKPOINT_INTERVAL
        )

        # Store results back in the main dataframe.
        for ki in range(K_NEAREST):
            col_m = f"{net_type}_dist_k{ki + 1}_m"
            col_km = f"{net_type}_dist_k{ki + 1}_km"
            sal_proj.loc[prov_mask_sal, col_m] = results[:, ki]
            sal_proj.loc[prov_mask_sal, col_km] = results[:, ki] / 1000.0

        run_elapsed = time.time() - run_start
        print(f"  {graph_key} COMPLETE: {run_elapsed / 60:.1f} min")
        print()

        # Save intermediate CSV after each province x mode combination.
        interim_path = os.path.join(OUTPUT_DIR, f"sal_distances_k{K_NEAREST}_interim_{label}.csv")
        interim_cols = ["EA_CODE"]
        for ki in range(K_NEAREST):
            interim_cols.extend([f"{net_type}_dist_k{ki + 1}_m", f"{net_type}_dist_k{ki + 1}_km"])
        available_interim = [c for c in interim_cols if c in sal_proj.columns]
        sal_proj.loc[prov_mask_sal, available_interim].to_csv(interim_path, index = False)
        print(f"  INTERIM SAVED: {interim_path}")
        print()

PROVINCE: Gauteng
  SALs: 20850, Pharmacies: 1511
  COMPUTING WALK NETWORK DISTANCES (k = 3)
  Pharmacy nodes snapped: 1511 total -> 1429 unique
  SAL centroid nodes snapped: 20850 total, 20607 unique
  PROCESSING 1429 remaining pharmacies (of 1429 total)
  PROGRESS: 50/1429 pharmacies (50/1429 this session) | avg 1.8s/pharm | ETA: 41 min
  PROGRESS: 100/1429 pharmacies (100/1429 this session) | avg 1.7s/pharm | ETA: 38 min
  PROGRESS: 150/1429 pharmacies (150/1429 this session) | avg 1.7s/pharm | ETA: 36 min
  PROGRESS: 200/1429 pharmacies (200/1429 this session) | avg 1.7s/pharm | ETA: 34 min
  PROGRESS: 250/1429 pharmacies (250/1429 this session) | avg 1.7s/pharm | ETA: 33 min
  PROGRESS: 300/1429 pharmacies (300/1429 this session) | avg 1.7s/pharm | ETA: 32 min
  PROGRESS: 350/1429 pharmacies (350/1429 this session) | avg 1.7s/pharm | ETA: 30 min
  PROGRESS: 400/1429 pharmacies (400/1429 this session) | avg 1.7s/pharm | ETA: 28 min
  PROGRESS: 450/1429 pharmacies (450/1429 this ses

## VALIDATE k=3 RESULTS

Confirm that k=1 <= k=2 <= k=3 (monotonicity check) and report
how many SALs have fewer than k pharmacies reachable.

In [19]:
# Monotonicity check: k=1 distance should always be <= k=2 <= k=3.
print("MONOTONICITY VALIDATION")
violations = 0

for mode in ["euclidean", "walk", "drive"]:
    for ki in range(1, K_NEAREST):
        col_prev = f"{mode}_dist_k{ki}_km"
        col_curr = f"{mode}_dist_k{ki + 1}_km"

        if col_prev not in sal_proj.columns or col_curr not in sal_proj.columns:
            continue

        # Only check where both values exist.
        mask = sal_proj[col_prev].notna() & sal_proj[col_curr].notna()
        if mask.sum() == 0:
            continue

        bad = sal_proj.loc[mask, col_curr] < sal_proj.loc[mask, col_prev]
        n_bad = bad.sum()
        violations += n_bad

        status = "PASS" if n_bad == 0 else f"FAIL ({n_bad} violations)"
        print(f"  {mode} k={ki} <= k={ki + 1}: {status}")

print(f"TOTAL VIOLATIONS: {violations}")
print()

# Reachability summary by k level.
print("REACHABILITY SUMMARY (SALs with valid distance)")
for mode in ["euclidean", "walk", "drive"]:
    for ki in range(1, K_NEAREST + 1):
        col = f"{mode}_dist_k{ki}_km"
        if col not in sal_proj.columns:
            continue
        valid = sal_proj[col].notna().sum()
        total = len(sal_proj)
        print(f"  {mode} k={ki}: {valid}/{total} ({valid / total * 100:.1f}%)")
    print()

MONOTONICITY VALIDATION
  euclidean k=1 <= k=2: PASS
  euclidean k=2 <= k=3: PASS
  walk k=1 <= k=2: PASS
  walk k=2 <= k=3: PASS
  drive k=1 <= k=2: PASS
  drive k=2 <= k=3: PASS
TOTAL VIOLATIONS: 0

REACHABILITY SUMMARY (SALs with valid distance)
  euclidean k=1: 38380/38380 (100.0%)
  euclidean k=2: 38380/38380 (100.0%)
  euclidean k=3: 38380/38380 (100.0%)

  walk k=1: 33449/38380 (87.2%)
  walk k=2: 29699/38380 (77.4%)
  walk k=3: 28691/38380 (74.8%)

  drive k=1: 33379/38380 (87.0%)
  drive k=2: 29720/38380 (77.4%)
  drive k=3: 28701/38380 (74.8%)



## DISTANCE SUMMARY STATISTICS

Compare distances across k levels, provinces, and distance modes.

In [20]:
# Summary statistics by province and k level.
for prov_name in PROVINCES:
    prov_mask = sal_proj[prov_col].str.contains(prov_name, case = False, na = False)
    subset = sal_proj[prov_mask]

    print(f"DISTANCE STATISTICS: {prov_name.upper()} (n = {len(subset)})")
    for mode in ["euclidean", "walk", "drive"]:
        for ki in range(1, K_NEAREST + 1):
            col = f"{mode}_dist_k{ki}_km"
            if col not in subset.columns:
                continue
            stats = subset[col].describe(percentiles = [0.5, 0.75, 0.9, 0.95])
            print(f"  {mode} k={ki}: median={stats['50%']:.2f} km, "
                  f"mean={stats['mean']:.2f} km, "
                  f"p90={stats['90%']:.2f} km, "
                  f"max={stats['max']:.2f} km")
        print()
    print()

DISTANCE STATISTICS: GAUTENG (n = 20850)
  euclidean k=1: median=0.91 km, mean=1.47 km, p90=2.93 km, max=35.52 km
  euclidean k=2: median=1.40 km, mean=2.11 km, p90=4.27 km, max=36.73 km
  euclidean k=3: median=1.75 km, mean=2.55 km, p90=5.12 km, max=36.75 km

  walk k=1: median=1.34 km, mean=2.03 km, p90=4.11 km, max=41.52 km
  walk k=2: median=2.02 km, mean=2.88 km, p90=5.76 km, max=42.23 km
  walk k=3: median=2.51 km, mean=3.48 km, p90=6.63 km, max=44.46 km

  drive k=1: median=1.32 km, mean=2.02 km, p90=4.16 km, max=38.11 km
  drive k=2: median=2.02 km, mean=2.87 km, p90=5.81 km, max=38.34 km
  drive k=3: median=2.53 km, mean=3.48 km, p90=6.69 km, max=42.85 km


DISTANCE STATISTICS: KWAZULU-NATAL (n = 17530)
  euclidean k=1: median=3.65 km, mean=7.95 km, p90=21.96 km, max=44.85 km
  euclidean k=2: median=5.70 km, mean=10.40 km, p90=27.15 km, max=60.48 km
  euclidean k=3: median=7.25 km, mean=12.73 km, p90=31.39 km, max=79.96 km

  walk k=1: median=9.66 km, mean=15.94 km, p90=40.12 

In [21]:
# Circuity ratios (network / euclidean) for k=1.
for net_col in ["walk_dist_k1_km", "drive_dist_k1_km"]:
    euc_col = "euclidean_dist_k1_km"
    if net_col in sal_proj.columns and euc_col in sal_proj.columns:
        ratio_col = net_col.replace("_dist_k1_km", "_circuity_k1")
        mask = (sal_proj[euc_col] > 0) & sal_proj[net_col].notna()
        sal_proj.loc[mask, ratio_col] = sal_proj.loc[mask, net_col] / sal_proj.loc[mask, euc_col]
        median_ratio = sal_proj[ratio_col].median()
        print(f"MEDIAN CIRCUITY RATIO ({net_col}): {median_ratio:.2f}")

MEDIAN CIRCUITY RATIO (walk_dist_k1_km): 1.55
MEDIAN CIRCUITY RATIO (drive_dist_k1_km): 1.53


## NETWORK DISTANCE HEATMAPS

Side-by-side choropleth maps comparing k=1 distances for Euclidean,
walk, and drive methods per province.

In [ ]:
# Heatmap for each distance type (k = 1 only for map clarity).
sal_map_data = sal_proj.copy()
sal_map_data = sal_map_data.set_geometry("geometry")

dist_bins_km = [0, 1, 2, 3, 5, 10, 15, 25, 50]
cmap = custom_cmap

# Distance columns to plot (k = 1 only).
plot_candidates = ["euclidean_dist_k1_km", "walk_dist_k1_km", "drive_dist_k1_km"]
available_cols = [c for c in plot_candidates if c in sal_map_data.columns]

for prov_name in PROVINCES:
    prov_mask = sal_map_data[prov_col].str.contains(prov_name, case = False, na = False)
    subset = sal_map_data[prov_mask].copy()

    plot_cols = [c for c in available_cols if subset[c].notna().sum() > 0]
    n_plots = len(plot_cols)

    if n_plots == 0:
        print(f"No distance data available for {prov_name}.")
        continue

    fig, axes = plt.subplots(1, n_plots, figsize = (7 * n_plots, 10))
    if n_plots == 1:
        axes = [axes]

    for j, col in enumerate(plot_cols):
        label = col.replace("_dist_k1_km", "").replace("_", " ").title()

        subset.plot(
            column = col,
            missing_kwds={"color": "#7d64ad", "label": "No Data"},
            cmap = cmap,
            scheme = "UserDefined",
            classification_kwds = {"bins": dist_bins_km},
            legend = (j == n_plots - 1),
            legend_kwds = {"title": "Distance (km)", "loc": "lower right", "fmt": "{:.0f}"},
            ax = axes[j],
            edgecolor = "none",
            linewidth = 0
        )

        axes[j].set_title(f"{prov_name} {label} Distance (k = 1)", fontweight = "bold", fontsize = 12)
        
        # Add scalebar to last subplot.
        if j == n_plots - 3:
            scalebar = ScaleBar(1, units = "m", location = "lower left", box_alpha = 0.8, font_properties = {"size": 10})
            axes[j].add_artist(scalebar)
        
        axes[j].axis("off")

    axes[j].get_legend().get_title().set_fontweight("bold")
    reformat_legend_labels(axes[j])
    plt.tight_layout(rect = [0, 0, 1, 0.92])
    #plt.suptitle(f"{prov_name}: Distance to Nearest Pharmacy by Method", fontweight = "bold", fontsize = 14, y = 0.97)

    output_path = os.path.join(IMAGE_DIR, f"heatmap_{prov_name.lower().replace('-', '_')}_all_methods_k1.png")

    #axes[j].set_facecolor("#fcffeb")
    plt.savefig(output_path, dpi = 300, bbox_inches = "tight")
    plt.savefig(f"{output_path}_transparent.png", dpi = 300, bbox_inches = "tight", transparent = True)
    print(f"SAVED: {output_path}")
    plt.show()

In [ ]:
# Heatmap for each distance type (k = 2).
sal_map_data = sal_proj.copy()
sal_map_data = sal_map_data.set_geometry("geometry")

dist_bins_km = [0, 1, 2, 3, 5, 10, 15, 25, 50]
cmap = custom_cmap

# Distance columns to plot (k = 2 only).
plot_candidates = ["euclidean_dist_k2_km", "walk_dist_k2_km", "drive_dist_k2_km"]
available_cols = [c for c in plot_candidates if c in sal_map_data.columns]

for prov_name in PROVINCES:
    prov_mask = sal_map_data[prov_col].str.contains(prov_name, case = False, na = False)
    subset = sal_map_data[prov_mask].copy()

    plot_cols = [c for c in available_cols if subset[c].notna().sum() > 0]
    n_plots = len(plot_cols)

    if n_plots == 0:
        print(f"No distance data available for {prov_name}.")
        continue

    fig, axes = plt.subplots(1, n_plots, figsize = (7 * n_plots, 10))
    if n_plots == 1:
        axes = [axes]

    for j, col in enumerate(plot_cols):
        label = col.replace("_dist_k2_km", "").replace("_", " ").title()

        subset.plot(
            column = col,
            missing_kwds={"color": "#7d64ad", "label": "No Data"},
            cmap = cmap,
            scheme = "UserDefined",
            classification_kwds = {"bins": dist_bins_km},
            legend = (j == n_plots - 1),
            legend_kwds = {"title": "Distance (km)", "loc": "lower right", "fmt": "{:.0f}"},
            ax = axes[j],
            edgecolor = "none",
            linewidth = 0
        )

        axes[j].set_title(f"{prov_name} {label} Distance (k = 2)", fontweight = "bold", fontsize = 12)
        
        # Add scalebar to last subplot.
        if j == n_plots - 3:
            scalebar = ScaleBar(1, units = "m", location = "lower left", box_alpha = 0.8, font_properties = {"size": 10})
            axes[j].add_artist(scalebar)
        
        axes[j].axis("off")

    axes[j].get_legend().get_title().set_fontweight("bold")
    reformat_legend_labels(axes[j])
    plt.tight_layout(rect = [0, 0, 1, 0.92])
    #plt.suptitle(f"{prov_name}: Distance to Nearest Pharmacy by Method", fontweight = "bold", fontsize = 14, y = 0.97)

    output_path = os.path.join(IMAGE_DIR, f"heatmap_{prov_name.lower().replace('-', '_')}_all_methods_k2.png")

    axes[j].set_facecolor("#fcffeb")
    plt.savefig(output_path, dpi = 300, bbox_inches = "tight")
    plt.savefig(f"{output_path}_transparent.png", dpi = 300, bbox_inches = "tight", transparent = True)
    print(f"SAVED: {output_path}")
    plt.show()

In [ ]:
# Heatmap for each distance type (k = 3).
sal_map_data = sal_proj.copy()
sal_map_data = sal_map_data.set_geometry("geometry")

dist_bins_km = [0, 1, 2, 3, 5, 10, 15, 25, 50]
cmap = custom_cmap

# Distance columns to plot (k = 3 only).
plot_candidates = ["euclidean_dist_k3_km", "walk_dist_k3_km", "drive_dist_k3_km"]
available_cols = [c for c in plot_candidates if c in sal_map_data.columns]

for prov_name in PROVINCES:
    prov_mask = sal_map_data[prov_col].str.contains(prov_name, case = False, na = False)
    subset = sal_map_data[prov_mask].copy()

    plot_cols = [c for c in available_cols if subset[c].notna().sum() > 0]
    n_plots = len(plot_cols)

    if n_plots == 0:
        print(f"No distance data available for {prov_name}.")
        continue

    fig, axes = plt.subplots(1, n_plots, figsize = (7 * n_plots, 10))
    if n_plots == 1:
        axes = [axes]

    for j, col in enumerate(plot_cols):
        label = col.replace("_dist_k3_km", "").replace("_", " ").title()

        subset.plot(
            column = col,
            missing_kwds={"color": "#7d64ad", "label": "No Data"},
            cmap = cmap,
            scheme = "UserDefined",
            classification_kwds = {"bins": dist_bins_km},
            legend = (j == n_plots - 1),
            legend_kwds = {"title": "Distance (km)", "loc": "lower right", "fmt": "{:.0f}"},
            ax = axes[j],
            edgecolor = "none",
            linewidth = 0
        )

        axes[j].set_title(f"{prov_name} {label} Distance (k = 3)", fontweight = "bold", fontsize = 12)
        
        # Add scalebar to last subplot.
        if j == n_plots - 3:
            scalebar = ScaleBar(1, units = "m", location = "lower left", box_alpha = 0.8, font_properties = {"size": 10})
            axes[j].add_artist(scalebar)
        
        axes[j].axis("off")

    axes[j].get_legend().get_title().set_fontweight("bold")
    reformat_legend_labels(axes[j])
    plt.tight_layout(rect = [0, 0, 1, 0.92])
    #plt.suptitle(f"{prov_name}: Distance to Nearest Pharmacy by Method", fontweight = "bold", fontsize = 14, y = 0.97)

    output_path = os.path.join(IMAGE_DIR, f"heatmap_{prov_name.lower().replace('-', '_')}_all_methods_k3.png")

    axes[j].set_facecolor("#fcffeb")
    plt.savefig(output_path, dpi = 300, bbox_inches = "tight")
    plt.savefig(f"{output_path}_transparent.png", dpi = 300, bbox_inches = "tight", transparent = True)
    print(f"SAVED: {output_path}")
    plt.show()

## NETWORK GRAPH PREVIEWS

Quick visual previews of the saved network graphs.

In [25]:
# # Plot a small preview of each saved network graph.
# for graph_key, G in graphs.items():
#     print(f"GRAPH: {graph_key}")
#     print(f"  Nodes: {G.number_of_nodes():,}")
#     print(f"  Edges: {G.number_of_edges():,}")

#     fig, ax = ox.plot_graph(
#         G,
#         figsize = (10, 10),
#         node_size = 0,
#         edge_linewidth = 0.3,
#         edge_color = "black",
#         bgcolor = "white",
#         show = False,
#         close = False
#     )

#     # Format title.
#     title_parts = graph_key.rsplit("_", 1)
#     if len(title_parts) == 2:
#         province, network = title_parts
#         title = f"{province} {network.capitalize()}"
#     else:
#         title = graph_key

#     ax.set_title("")

#     output_path = os.path.join(IMAGE_DIR, f"network_{graph_key.lower().replace('-', '_')}.png")

#     fig.patch.set_alpha(0.0)
#     ax.set_facecolor("none")
#     fig.savefig(output_path, dpi = 150, bbox_inches = "tight", facecolor = "white")
#     plt.show()
#     print(f"SAVED: {output_path}")
#     print()

## EXPORT RESULTS

Save the SAL-level distance results for downstream analysis.
Output includes k=1, k=2, k=3 distances for all three methods.

In [26]:
# Build export column list.
export_cols = ["EA_CODE", "WardID", "sal2023_est",
               "centroid_lat", "centroid_lng"]

# Add province column.
if prov_col in sal_proj.columns:
    export_cols.insert(1, prov_col)

# Add Euclidean distance columns for all k levels.
for ki in range(1, K_NEAREST + 1):
    for suffix in ["_m", "_km"]:
        col = f"euclidean_dist_k{ki}{suffix}"
        if col in sal_proj.columns:
            export_cols.append(col)

# Add network distance columns for all k levels.
for net_type in ["walk", "drive"]:
    for ki in range(1, K_NEAREST + 1):
        for suffix in ["_m", "_km"]:
            col = f"{net_type}_dist_k{ki}{suffix}"
            if col in sal_proj.columns:
                export_cols.append(col)

# Add circuity columns.
for col in sal_proj.columns:
    if "circuity" in col:
        export_cols.append(col)

# Filter to only columns that exist.
export_cols = [c for c in export_cols if c in sal_proj.columns]

export_df = sal_proj[export_cols].copy()
export_path = os.path.join(OUTPUT_DIR, f"sal_pharmacy_distances_k{K_NEAREST}.csv")
export_df.to_csv(export_path, index = False)

print(f"RESULTS EXPORTED: {export_path}")
print(f"Rows: {export_df.shape[0]}, Columns: {export_df.shape[1]}")
print(f"Columns: {list(export_df.columns)}")

RESULTS EXPORTED: data/networks\sal_pharmacy_distances_k3.csv
Rows: 38380, Columns: 26
Columns: ['EA_CODE', 'PR_NAME', 'WardID', 'sal2023_est', 'centroid_lat', 'centroid_lng', 'euclidean_dist_k1_m', 'euclidean_dist_k1_km', 'euclidean_dist_k2_m', 'euclidean_dist_k2_km', 'euclidean_dist_k3_m', 'euclidean_dist_k3_km', 'walk_dist_k1_m', 'walk_dist_k1_km', 'walk_dist_k2_m', 'walk_dist_k2_km', 'walk_dist_k3_m', 'walk_dist_k3_km', 'drive_dist_k1_m', 'drive_dist_k1_km', 'drive_dist_k2_m', 'drive_dist_k2_km', 'drive_dist_k3_m', 'drive_dist_k3_km', 'walk_circuity_k1', 'drive_circuity_k1']


In [27]:
# Also export a backward-compatible k=1-only CSV matching the original format.
compat_cols = ["EA_CODE"]
if prov_col in sal_proj.columns:
    compat_cols.append(prov_col)

compat_cols.extend(["WardID", "sal2023_est", "centroid_lat", "centroid_lng"])

# Map k=1 columns to original names for compatibility.
rename_map = {
    "euclidean_dist_k1_m": "euclidean_dist_m",
    "euclidean_dist_k1_km": "euclidean_dist_km",
    "walk_dist_k1_m": "walk_dist_m",
    "walk_dist_k1_km": "walk_dist_km",
    "drive_dist_k1_m": "drive_dist_m",
    "drive_dist_k1_km": "drive_dist_km",
}

for new_name, old_name in rename_map.items():
    if new_name in sal_proj.columns:
        compat_cols.append(new_name)

compat_cols = [c for c in compat_cols if c in sal_proj.columns]
compat_df = sal_proj[compat_cols].copy()
compat_df = compat_df.rename(columns = rename_map)

compat_path = os.path.join(OUTPUT_DIR, "sal_pharmacy_distances.csv")
compat_df.to_csv(compat_path, index = False)
print(f"BACKWARD-COMPATIBLE EXPORT: {compat_path}")
print(f"(k=1 columns renamed to match original column names for existing notebooks)")

BACKWARD-COMPATIBLE EXPORT: data/networks\sal_pharmacy_distances.csv
(k=1 columns renamed to match original column names for existing notebooks)


## CHECKPOINT CLEANUP

Verify no orphaned checkpoint files remain after successful
completion. If any exist, the corresponding computation did
not finish cleanly.

In [28]:
# Check for any remaining checkpoint files.
remaining_checkpoints = [f for f in os.listdir(CHECKPOINT_DIR) if f.endswith(".pkl")]

if remaining_checkpoints:
    print(f"WARNING: {len(remaining_checkpoints)} checkpoint file(s) found.")
    print("These indicate interrupted computations that did not complete:")
    for f in remaining_checkpoints:
        fpath = os.path.join(CHECKPOINT_DIR, f)
        with open(fpath, "rb") as fh:
            cp = pickle.load(fh)
        print(f"  {f}: {len(cp['processed'])} pharmacies processed")
    print()
    print("Re-run the computation cell to resume from these checkpoints.")
else:
    print("NO CHECKPOINT FILES FOUND: all computations completed successfully.")

NO CHECKPOINT FILES FOUND: all computations completed successfully.


## NOTES

- SAL centroids are geometric (unweighted), and for irregularly shaped
  or large rural SALs, the centroid may fall in uninhabited areas,
  introducing measurement error. Population-weighted centroids using
  building footprints would reduce this bias.

- OSM road and pedestrian networks may have coverage gaps in rural and
  informal settlement areas, so missing roads mean the algorithm either
  routes through longer detours or marks centroids as unreachable.

- Drive networks are directed, meaning that one-way streets matter. The
  code converts to undirected for Dijkstra, which simplifies routing but
  may slightly underestimate true one-way-constrained drive distances.

- The walk network includes sidewalks, footpaths, and pedestrian-accessible
  roads, but in practice pedestrians in informal settlements may use paths
  not mapped in OSM.

- Circuity is the ratio of network distance to Euclidean distance, which is
  a useful diagnostic. Ratios of 1.2-1.5 are typical for urban grids, and
  higher ratios suggest barriers (rivers, highways, rail lines) or sparse
  road networks. Extremely high ratios (>3) may indicate network data gaps.

- Both centroids and pharmacies are snapped to the nearest network node.
  For pharmacies near highways, this may snap to a highway node rather than
  the actual pedestrian-accessible entrance. Adding a maximum snap tolerance
  (e.g. 500m) would flag pharmacies or centroids far from any mapped road.

- **k=3 compute cost:** The single-source Dijkstra approach is O(P x V log V)
  per graph, where P = unique pharmacy nodes and V = reachable graph nodes
  within the cutoff. For province-scale graphs with thousands of pharmacies,
  this can take several hours per graph. The checkpoint system ensures that
  interrupted runs resume without losing progress.

- **Cutoff trade-off:** The 50 km Dijkstra cutoff keeps computation bounded.
  SALs with no pharmacy within 50 km by network will show NaN for that mode.
  This is acceptable because a 50 km network distance represents a travel
  time well beyond any reasonable access threshold.